In [36]:
!pip install -q datasets huggingface_hub pandas tqdm

# IndicVoices Dataset Exploration

Internship Project

Objective:
- Read IndicVoices dataset
- Filter required languages
- Ensure speaker independence
- Create train/validation/test CSV files

In [39]:
import random
from collections import defaultdict

import pandas as pd
from tqdm import tqdm

from datasets import load_dataset
from huggingface_hub import notebook_login

In [40]:
from huggingface_hub import notebook_login

notebook_login()

In [41]:
LANGUAGE = "bengali"

TRAIN_SAMPLES = 5000
VALID_SAMPLES = 1000
TEST_SAMPLES = 1000

SAMPLES_PER_SPEAKER = 10

TOTAL_SAMPLES = (
    TRAIN_SAMPLES +
    VALID_SAMPLES +
    TEST_SAMPLES
)

TOTAL_SPEAKERS = TOTAL_SAMPLES // SAMPLES_PER_SPEAKER

print("Language:", LANGUAGE)
print("Total speakers required:", TOTAL_SPEAKERS)

Language: bengali
Total speakers required: 700


In [42]:
dataset = load_dataset(
    "ai4bharat/IndicVoices",
    LANGUAGE,
    split="train",
    streaming=True
)

print(dataset)

Resolving data files:   0%|          | 0/112 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/96 [00:00<?, ?it/s]

IterableDataset({
    features: ['audio_filepath', 'text', 'duration', 'lang', 'samples', 'verbatim', 'normalized', 'speaker_id', 'scenario', 'task_name', 'gender', 'age_group', 'job_type', 'qualification', 'area', 'district', 'state', 'occupation', 'verification_report', 'unsanitized_verbatim', 'unsanitized_normalized'],
    num_shards: 96
})


In [43]:
sample = next(iter(dataset))

print("Keys:")
print(sample.keys())

print("\nSpeaker ID:", sample["speaker_id"])
print("Language  :", sample["lang"])
print("Gender    :", sample["gender"])
print("Age Group :", sample["age_group"])
print("State     :", sample["state"])
print("District  :", sample["district"])

print("\nText:")
print(sample["text"])

print("\nAudio type:")
print(type(sample["audio_filepath"]))

print("\nAudio object:")
print(sample["audio_filepath"])

HfHubHTTPError: (Request ID: 01KXGVP6YBR7607JP4D12QH3EW)

403 Forbidden: None.
Cannot access content at: https://us.gcp.cdn.hf.co/xet-bridge-us/67c940995477d45b871a8f1c/02657c4f9c30561be4b428fcb8dc2844279584508dc85db40f55f4b3dfbe2ad2?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27train-00000-of-00096.parquet%3B+filename%3D%22train-00000-of-00096.parquet%22%3B&X-Xet-Cas-Uid=6a47fed3c458bfb483d32944&user_id=6a47fed3c458bfb483d32944&Expires=1784054680&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjdjOTQwOTk1NDc3ZDQ1Yjg3MWE4ZjFjLzAyNjU3YzRmOWMzMDU2MWJlNGI0MjhmY2I4ZGMyODQ0Mjc5NTg0NTA4ZGM4NWRiNDBmNTVmNGIzZGZiZTJhZDJcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomWC1YZXQtQ2FzLVVpZD02YTQ3ZmVkM2M0NThiZmI0ODNkMzI5NDQmdXNlcl9pZD02YTQ3ZmVkM2M0NThiZmI0ODNkMzI5NDQiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkVwb2NoVGltZSI6MTc4NDA1NDY4MH0sIkJ5dGVSYW5nZSI6eyJFeHBlY3RlZEhlYWRlciI6ImJ5dGVzPTQ4NjU4NzU3NS00ODY2NTMxMTAifX19XX0_&Signature=MEQCIHS34E

In [ ]:
sample = next(iter(dataset))

audio = sample["audio_filepath"]

print(type(audio))
print(audio)

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "ai4bharat/IndicVoices",
    "hindi",
    split="train",
    streaming=True
)

# IMPORTANT: disable audio decoding
dataset = dataset.decode(False)

print(dataset)

In [ ]:
from datasets import get_dataset_config_names

configs = get_dataset_config_names("ai4bharat/IndicVoices")

print(configs)

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

files = api.list_repo_files(
    repo_id="ai4bharat/IndicVoices",
    repo_type="dataset"
)

print("Total files:", len(files))

for f in files[:100]:
    print(f)

In [ ]:
# Search for Hindi parquet files

for f in files:
    if f.startswith("bengali/"):
        print(f)

        # print only first 15
        if "00014" in f:
            break

In [ ]:
from datasets import load_dataset

# Read ONLY ONE parquet shard
parquet = load_dataset(
    "parquet",
    data_files="hf://datasets/ai4bharat/IndicVoices/bengali/train-00000-of-00082.parquet"
)

print(parquet)

In [ ]:

from datasets import load_dataset
from collections import Counter
import gc

speaker_counts = Counter()

for shard in range(82):

    print(f"\nProcessing shard {shard+1}/82")

    filename = f"hf://datasets/ai4bharat/IndicVoices/bengali/train-{shard:05d}-of-00082.parquet"

    parquet = load_dataset(
        "parquet",
        data_files=filename
    )

    for speaker in parquet["train"]["speaker_id"]:
        speaker_counts[speaker] += 1

    del parquet
    gc.collect()   # Force memory cleanup after every shard

print("Done!")
print("Unique speakers:", len(speaker_counts))

In [ ]:
eligible_speakers = {
    sp: cnt
    for sp, cnt in speaker_counts.items()
    if cnt >= 10
}

print("Speakers with >=10 recordings:", len(eligible_speakers))

In [ ]:
selected_speakers = list(eligible_speakers.keys())[:700]

print(len(selected_speakers))

In [ ]:
import json

with open("/kaggle/working/selected_speakers.json", "w") as f:
    json.dump(selected_speakers, f)

print("Saved!")

In [ ]:
with open("/kaggle/working/eligible_speakers.json", "w") as f:
    json.dump(eligible_speakers, f)

print("Eligible speakers saved!")

In [ ]:
with open("/kaggle/working/speaker_counts.json", "w") as f:
    json.dump(speaker_counts, f)

print("Speaker counts saved!")

In [ ]:
selected_speakers = [
    sp for sp, cnt in sorted(
        eligible_speakers.items(),
        key=lambda x: x[1],
        reverse=True
    )[:700]
]

print("Selected speakers:", len(selected_speakers))

In [ ]:
import random

random.seed(42)

random.shuffle(selected_speakers)

train_speakers = set(selected_speakers[:500])
valid_speakers = set(selected_speakers[500:600])
test_speakers = set(selected_speakers[600:700])

print("Train speakers :", len(train_speakers))
print("Validation speakers :", len(valid_speakers))
print("Test speakers :", len(test_speakers))

In [ ]:
import json

with open("/kaggle/working/train_speakers.json", "w") as f:
    json.dump(list(train_speakers), f)

with open("/kaggle/working/valid_speakers.json", "w") as f:
    json.dump(list(valid_speakers), f)

with open("/kaggle/working/test_speakers.json", "w") as f:
    json.dump(list(test_speakers), f)

print("Speaker splits saved successfully!")

In [ ]:
with open("/kaggle/working/selected_speakers.json", "w") as f:
    json.dump(selected_speakers, f)

print("Selected speakers saved!")

In [ ]:
from datasets import load_dataset
from collections import defaultdict
import os
import csv
import soundfile as sf
from io import BytesIO
import gc

In [ ]:
import os

base_dir = f"/kaggle/working/IndicVoices700_{LANGUAGE}"

os.makedirs(f"{base_dir}/train/audio", exist_ok=True)
os.makedirs(f"{base_dir}/valid/audio", exist_ok=True)
os.makedirs(f"{base_dir}/test/audio", exist_ok=True)

print(f"Created folders for {LANGUAGE}")

In [ ]:
speaker_recordings = defaultdict(int)

train_rows = []
valid_rows = []
test_rows = []

TARGET = 10

In [ ]:
sample = next(iter(dataset))

audio = sample["audio_filepath"]

print(type(audio))
print(audio)

In [ ]:
from datasets import load_dataset
from collections import defaultdict
import pandas as pd
import gc

speaker_count = defaultdict(int)

rows = []

In [ ]:
sample = ds[0]

audio = sample["audio_filepath"]

print(type(audio))

print(dir(audio)[:50])

In [ ]:
print(hasattr(audio, "get_all_samples"))
print(hasattr(audio, "array"))
print(hasattr(audio, "sampling_rate"))

In [ ]:
sample = ds[0]

decoded = sample["audio_filepath"].get_all_samples()

print(type(decoded))
print(dir(decoded))

print(decoded.sample_rate)
print(decoded.data.shape)

In [ ]:
!pip install soundfile pandas -q

In [ ]:
for shard in range(82):

    print(f"\nProcessing shard {shard+1}/82")

    filename = f"hf://datasets/ai4bharat/IndicVoices/bengali/train-{shard:05d}-of-00082.parquet"

    ds = load_dataset(
        "parquet",
        data_files=filename,
        split="train"
    )

    for sample in ds:

        speaker = sample["speaker_id"]

        if speaker not in selected_speakers:
            continue

        if speaker_counter[speaker] >= TARGET:
            continue

        decoded = sample["audio_filepath"].get_all_samples()

        waveform = decoded.data.squeeze().numpy()
        sr = decoded.sample_rate

        if speaker in train_speakers:
            split = "train"
        elif speaker in valid_speakers:
            split = "valid"
        else:
            split = "test"

        filename = f"{speaker}_{speaker_counter[speaker]}.wav"

        save_path = os.path.join(
            base_dir,
            split,
            filename
        )

        sf.write(save_path, waveform, sr)

        row = {
            "speaker_id": speaker,
            "audio": filename,
            "text": sample["text"],
            "gender": sample["gender"],
            "state": sample["state"],
            "district": sample["district"]
        }

        if split == "train":
            train_rows.append(row)
        elif split == "valid":
            valid_rows.append(row)
        else:
            test_rows.append(row)

        speaker_counter[speaker] += 1

    del ds
    gc.collect()

    print("Collected:", sum(speaker_counter.values()))

    if sum(speaker_counter.values()) >= 7000:
        print("Done collecting!")
        break

In [ ]:
print(len(train_speakers))
print(len(valid_speakers))
print(len(test_speakers))

print(os.listdir("/kaggle/working/IndicVoices700_bengali"))

In [ ]:
print(len(train_rows))
print(len(valid_rows))
print(len(test_rows))

In [ ]:
import pandas as pd

pd.DataFrame(train_rows).to_csv(
    f"{base_dir}/train_metadata.csv",
    index=False
)

pd.DataFrame(valid_rows).to_csv(
    f"{base_dir}/valid_metadata.csv",
    index=False
)

pd.DataFrame(test_rows).to_csv(
    f"{base_dir}/test_metadata.csv",
    index=False
)

print("Metadata saved!")

In [ ]:
import os

print("Train:", len(os.listdir(f"{base_dir}/train")))
print("Valid:", len(os.listdir(f"{base_dir}/valid")))
print("Test :", len(os.listdir(f"{base_dir}/test")))

In [ ]:
import shutil

shutil.make_archive(
    "/kaggle/working/IndicVoices700_bengali",
    "zip",
    base_dir
)

print("Zip created!")

In [ ]:
import shutil

shutil.make_archive("/kaggle/working/train_bengali", "zip",
                    "/kaggle/working/IndicVoices700/train")

shutil.make_archive("/kaggle/working/valid_bengali", "zip",
                    "/kaggle/working/IndicVoices700/valid")

shutil.make_archive("/kaggle/working/test_bengali", "zip",
                    "/kaggle/working/IndicVoices700/test")

print("Done!")